In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler,LabelEncoder
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import accuracy_score,mean_squared_error
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline,make_pipeline

In [ ]:
from google.colab import files
import pandas as pd

uploaded = files.upload()

In [ ]:
df = pd.read_csv("car_price_prediction_.csv")
df.head()

In [ ]:
df.head().style.background_gradient(cmap='coolwarm', axis=0)

In [ ]:
!pip install colorama
from colorama import Fore, Style

# Print the shape of the dataframe (number of rows and columns)
print(Fore.CYAN + "df shape: " + Style.RESET_ALL)
print(f"{df.shape}\n")

# Print basic information about the dataframe (column names, data types, non-null values)
print(Fore.GREEN + "df info: " + Style.RESET_ALL)
print(f"{df.info()}\n")

# Print the count of missing (NaN) values in each column
print(Fore.YELLOW + "df isnull sum: " + Style.RESET_ALL)
print(f"{df.isnull().sum()}\n")

# Print summary statistics for numerical columns (count, mean, std, min, max, etc.)
print(Fore.MAGENTA + "df describe: " + Style.RESET_ALL)
print(f"{df.describe()}\n")

In [ ]:
df.drop(columns=["Car ID"],inplace=True) # Drop the "Car Id" column from the dataframe

In [ ]:
catgorical_values = df.select_dtypes(include=["object"])

for catgorical in catgorical_values:
    counts = df[catgorical].value_counts()
    plt.figure(figsize = (12,5))
    plt.subplot(1,2,1)
    sns.countplot(data = df, x = catgorical, palette = "Set2")
    plt.title(f"Count of {catgorical} values")
    plt.xticks(rotation = 90)
    plt.ylabel("Count")
    # plt.show()

    plt.subplot(1,2,2)
    # plt.figure(figsize = (12,5))
    plt.pie(counts,labels=counts.index,autopct='%1.1f%%',startangle=90)
    plt.title(f"Percentage of {catgorical} values")
    plt.axis('equal')  # Equal aspect ratio ensures that pie is drawn as a circle.
    plt.tight_layout()
    plt.show()
    print("\n")

In [ ]:
num_values = df.select_dtypes(include = ["int64"])
for num in num_values:
    plt.figure(figsize = (12,5))
    plt.subplot(1,2,1)
    sns.histplot(data = df,x = num,kde = True,palette = "Set2",bins=20)
    plt.title(f"Distribution of {num} values")
    plt.xlabel(num)
    plt.ylabel("Count")
    # plt.show()

    plt.subplot(1,2,2)
    sns.boxplot(data = df,x = num,palette = "Set2")
    plt.title(f"Boxplot of {num} values")
    plt.xlabel(num)
    # plt.show()
    plt.tight_layout()
    plt.show()

In [ ]:
# Group the data by "Brand" and calculate the mean price, then sort in descending order and plot as a bar chart
df.groupby("Brand")["Price"].mean().sort_values(ascending=False).plot(kind="bar", figsize=(12, 5), color="orange")
plt.title("Average Price of Cars by Brand")
plt.ylabel("Average Price")  # Label for the y-axis
plt.xlabel("Brand")  # Label for the x-axis
plt.xticks(rotation=90)

# Group the data by "Brand" and calculate the max price, then sort in descending order and plot as a bar chart
df.groupby("Brand")["Price"].max().sort_values(ascending=False).plot(kind="bar", figsize=(12, 5), color="red")
plt.title("Max Price of Cars by Brand")
plt.ylabel("Max Price")  # Label for the y-axis
plt.xlabel("Brand")  # Label for the x-axis
plt.xticks(rotation=90)
plt.show()

In [ ]:
df.groupby(["Transmission","Brand"])["Price"].mean().sort_values(ascending=False).reset_index().pivot(index="Brand",columns="Transmission",values="Price").plot(kind="bar", figsize=(12, 5), color=["orange","red"])
plt.title("Average Price of Cars by Brand and Transmission")
plt.ylabel("Average Price")  # Label for the y-axis
plt.xlabel("Brand")  # Label for the x-axis
plt.xticks(rotation=90)
plt.legend(title="Transmission")
plt.show()

In [ ]:
x = df.drop(columns=["Price"])
y = df["Price"]

# split the data into training and testing sets (80% training, 20% testing)
x_train, x_test, y_train, y_test = train_test_split(x,y,test_size=0.2,random_state=42)

numerical_features = x.select_dtypes(include=["int64","float64"]).columns.tolist()
catogorical_features = x.select_dtypes(include=["object"]).columns.tolist()

numerical_transformer = make_pipeline(
    StandardScaler()
)



catogorical_transformer = make_pipeline(
    OneHotEncoder(handle_unknown='ignore')
)

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numerical_transformer, numerical_features),
        ('cat', catogorical_transformer, catogorical_features)
    ],
    remainder='passthrough'
)

# Create a pipeline for the model
models = [
    ["LinearRegression:",LinearRegression()],
    ['DesisionTreeRegressor:',DecisionTreeRegressor()],
    ["RandomFrorestRegressor:",RandomForestRegressor()]
]

# Evaluate each model
reg_pred = []

for name, model in models:
    pipe = make_pipeline(preprocessor, model)
    pipe.fit(x_train, y_train)
    predictions = pipe.predict(x_test)
    rms = np.sqrt(mean_squared_error(y_test, predictions))
    reg_pred.append(rms)
    print(f" {name} RMSE: {rms:.2f}")

y_ax = ["Linear Regression","DecisionTreeRegressor","RandomForestRegressor"]
x_ax = reg_pred

sns.barplot(x=x_ax,y=y_ax,linewidth=1.5,edgecolor="0.1",color="orange")
plt.title("RMSE of Different Models")
plt.show()

In [ ]:
x = df.drop(columns=["Price"])
y = df["Price"]

# split the data into training and testing sets (80% training, 20% testing)
x_train, x_test, y_train, y_test = train_test_split(x,y,test_size=0.2,random_state=42)

numerical_features = x.select_dtypes(include=["int64","float64"]).columns.tolist()
catogorical_features = x.select_dtypes(include=["object"]).columns.tolist()

numerical_transformer = make_pipeline(
    StandardScaler()
)

catogorical_transformer = make_pipeline(
    OneHotEncoder(handle_unknown='ignore')
)

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numerical_transformer, numerical_features),
        ('cat', catogorical_transformer, catogorical_features)
    ],
    remainder='passthrough'
)


# Models and Parameter Grids
models_and_params = [
    ("LinearRegression:",LinearRegression(),{}),
    ("DecisionTreeRegressor:",DecisionTreeRegressor(),{
        'decisiontreeregressor__max_depth': [3, 5, 10, None],
        'decisiontreeregressor__min_samples_split': [2, 5, 10]
    }),
    ("RandomForestRegressor:",RandomForestRegressor(),{
        'randomforestregressor__n_estimators': [10, 50, 100],
        'randomforestregressor__max_depth': [3, 5, 10, None],
        'randomforestregressor__min_samples_split': [2, 5, 10]
    })
]

reg_pred = []
model_names = []

for name, model, param_grid in models_and_params:
    pipe = make_pipeline(preprocessor, model)

    # If there are hyperparameters to tune
    if param_grid:
        grid = GridSearchCV(pipe, param_grid, cv=5, scoring='neg_mean_squared_error', n_jobs=-1)
        grid.fit(x_train, y_train)
        best_model = grid.best_estimator_
        print(f"Best parameters for {name}: {grid.best_params_}")
    else:
        best_model = pipe
        best_model.fit(x_train, y_train)
    predictions = best_model.predict(x_test)
    rms = np.sqrt(mean_squared_error(y_test, predictions))
    reg_pred.append(rms)
    model_names.append(name)
    print(f" {name} RMSE: {rms:.2f}")

plt.figure(figsize=(10, 6))
sns.barplot(x=model_names, y=reg_pred, palette="Set2", linewidth=1.5, edgecolor="0.1")
plt.title("RMSE of Different Models with Hyperparameter Tuning")
plt.ylabel("RMSE")
plt.xlabel("Models")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

for column in df.select_dtypes(include="object").columns:
    df[column] = LabelEncoder().fit_transform(df[column])

X = df.drop("Price", axis=1)
y = df["Price"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model = RandomForestRegressor(random_state=42)
model.fit(X_train, y_train)

predictions = model.predict(X_test)

print("MAE:", mean_absolute_error(y_test, predictions))
print("RMSE:", np.sqrt(mean_squared_error(y_test, predictions)))
print("R² Score:", r2_score(y_test, predictions))

In [ ]:
plt.scatter(y_test, predictions)
plt.xlabel("Actual Price")
plt.ylabel("Predicted Price")
plt.title("Actual vs Predicted Car Prices")
plt.show()